In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from tqdm import tqdm

In [2]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...


True

In [3]:
def find_project_root(start_path: Path) -> Path:
    """
    Finds the project root folder by checking for requirements.txt and data folder.
    This works even if the notebook runs from notebooks/ or project root.
    """
    start_path = start_path.resolve()
    
    for path in [start_path] + list(start_path.parents):
        if (path / "requirements.txt").exists() and (path / "data").exists():
            return path
    
    return start_path


PROJECT_ROOT = find_project_root(Path.cwd())

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

COMBINED_DATA_PATH = PROCESSED_DIR / "combined_news.csv"
CLEANED_DATA_PATH = PROCESSED_DIR / "cleaned_news.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data folder:", PROCESSED_DIR)

Project root: C:\Users\user\Documents\fake news detection 0.1\FakeNewsDetector
Processed data folder: C:\Users\user\Documents\fake news detection 0.1\FakeNewsDetector\data\processed


In [4]:
def load_dataset() -> pd.DataFrame:
    """
    Loads the dataset for preprocessing.
    Priority:
    1. data/processed/combined_news.csv
    2. data/fake.csv and data/real.csv
    """
    
    if COMBINED_DATA_PATH.exists():
        print("Loading combined dataset:", COMBINED_DATA_PATH)
        data = pd.read_csv(COMBINED_DATA_PATH)
        return data
    
    fake_path = DATA_DIR / "fake.csv"
    real_path = DATA_DIR / "real.csv"
    
    if not fake_path.exists() or not real_path.exists():
        raise FileNotFoundError(
            "Dataset files not found. Please check:\n"
            f"{fake_path}\n"
            f"{real_path}\n"
            "or create data/processed/combined_news.csv first."
        )
    
    print("Loading raw dataset files...")
    fake_news = pd.read_csv(fake_path)
    real_news = pd.read_csv(real_path)
    
    fake_news["label"] = 0
    real_news["label"] = 1
    
    data = pd.concat([fake_news, real_news], ignore_index=True)
    
    return data


news_data = load_dataset()

print("Dataset shape:", news_data.shape)
display(news_data.head())

Loading raw dataset files...
Dataset shape: (44898, 5)


,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0


In [5]:
news_data = news_data.copy()

# If content column already exists, use it.
# Otherwise, combine title and text columns.
if "content" not in news_data.columns:
    if "title" in news_data.columns and "text" in news_data.columns:
        news_data["title"] = news_data["title"].fillna("")
        news_data["text"] = news_data["text"].fillna("")
        news_data["content"] = news_data["title"] + " " + news_data["text"]
    elif "text" in news_data.columns:
        news_data["content"] = news_data["text"].fillna("")
    else:
        raise ValueError("No valid text column found. Expected 'content' or 'text' column.")

# Keep only required columns
news_data = news_data[["content", "label"]]

print("Columns after selection:")
print(news_data.columns)

display(news_data.head())

Columns after selection:
Index(['content', 'label'], dtype='str')


,content,label
0,Donald Trump Sends Out Embarrassing New Year’...,0
1,Drunk Bragging Trump Staffer Started Russian ...,0
2,Sheriff David Clarke Becomes An Internet Joke...,0
3,Trump Is So Obsessed He Even Has Obama’s Name...,0
4,Pope Francis Just Called Out Donald Trump Dur...,0


In [6]:
print("Before validation")
print("Dataset shape:", news_data.shape)
print("\nMissing values:")
print(news_data.isnull().sum())

print("\nClass distribution:")
print(news_data["label"].value_counts())

Before validation
Dataset shape: (44898, 2)

Missing values:
content    0
label      0
dtype: int64

Class distribution:
label
0    23481
1    21417
Name: count, dtype: int64


In [7]:
# Remove missing content
news_data["content"] = news_data["content"].fillna("")

# Remove empty text records
news_data["content"] = news_data["content"].astype(str).str.strip()
news_data = news_data[news_data["content"] != ""]

# Remove duplicate articles
before_duplicates = len(news_data)
news_data = news_data.drop_duplicates(subset=["content"])
after_duplicates = len(news_data)

# Ensure label is integer
news_data["label"] = news_data["label"].astype(int)

print("After validation")
print("Dataset shape:", news_data.shape)
print("Removed duplicates:", before_duplicates - after_duplicates)

print("\nClass distribution:")
print(news_data["label"].value_counts())

print("\nMissing values:")
print(news_data.isnull().sum())

After validation
Dataset shape: (39103, 2)
Removed duplicates: 5795

Class distribution:
label
1    21196
0    17907
Name: count, dtype: int64

Missing values:
content    0
label      0
dtype: int64


In [8]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text: str) -> str:
    """
    Clean and preprocess news text.
    
    Steps:
    1. Convert to lowercase
    2. Remove URLs
    3. Remove HTML tags
    4. Remove non-alphabetic characters
    5. Remove extra spaces
    6. Remove stop words
    7. Lemmatize words
    """
    
    text = str(text)
    
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    
    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)
    
    # Remove non-alphabetic characters
    text = re.sub(r"[^a-z\s]", " ", text)
    
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    # Tokenization using split
    tokens = text.split()
    
    # Remove stop words and short words
    tokens = [
        word for word in tokens
        if word not in stop_words and len(word) > 2
    ]
    
    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return " ".join(tokens)

In [10]:
sample_text = news_data["content"].iloc[0]

print("Original text:")
print(sample_text[:500])

print("\nCleaned text:")
print(clean_text(sample_text)[:500])

Original text:
Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a

Cleaned text:
donald trump sends embarrassing new year eve message disturbing donald trump wish american happy new year leave instead give shout enemy hater dishonest fake news medium former reality show star one job country rapidly grows stronger smarter want wish friend supporter enemy hater even dishonest fake news medium happy healthy new year president angry pant tweeted great year america country rapidly grows stronger smarter want wish friend supporter enemy hater even di

In [11]:
tqdm.pandas()

news_data["clean_text"] = news_data["content"].progress_apply(clean_text)

display(news_data.head())

100%|██████████| 39103/39103 [03:54<00:00, 166.79it/s]


,content,label,clean_text
0,Donald Trump Sends Out Embarrassing New Year’s...,0,donald trump sends embarrassing new year eve m...
1,Drunk Bragging Trump Staffer Started Russian C...,0,drunk bragging trump staffer started russian c...
2,Sheriff David Clarke Becomes An Internet Joke ...,0,sheriff david clarke becomes internet joke thr...
3,Trump Is So Obsessed He Even Has Obama’s Name ...,0,trump obsessed even obama name coded website i...
4,Pope Francis Just Called Out Donald Trump Duri...,0,pope francis called donald trump christmas spe...


In [12]:
before_empty_clean = len(news_data)

news_data["clean_text"] = news_data["clean_text"].fillna("").astype(str).str.strip()
news_data = news_data[news_data["clean_text"] != ""]

after_empty_clean = len(news_data)

print("Removed empty cleaned records:", before_empty_clean - after_empty_clean)
print("Final dataset shape:", news_data.shape)

print("\nFinal class distribution:")
print(news_data["label"].value_counts())

Removed empty cleaned records: 5
Final dataset shape: (39098, 3)

Final class distribution:
label
1    21196
0    17902
Name: count, dtype: int64


In [13]:
final_data = news_data[["content", "clean_text", "label"]]

final_data.to_csv(CLEANED_DATA_PATH, index=False)

print("Cleaned dataset saved successfully!")
print("File path:", CLEANED_DATA_PATH)

display(final_data.head())

Cleaned dataset saved successfully!
File path: C:\Users\user\Documents\fake news detection 0.1\FakeNewsDetector\data\processed\cleaned_news.csv


,content,clean_text,label
0,Donald Trump Sends Out Embarrassing New Year’s...,donald trump sends embarrassing new year eve m...,0
1,Drunk Bragging Trump Staffer Started Russian C...,drunk bragging trump staffer started russian c...,0
2,Sheriff David Clarke Becomes An Internet Joke ...,sheriff david clarke becomes internet joke thr...,0
3,Trump Is So Obsessed He Even Has Obama’s Name ...,trump obsessed even obama name coded website i...,0
4,Pope Francis Just Called Out Donald Trump Duri...,pope francis called donald trump christmas spe...,0


In [14]:
cleaned_data = pd.read_csv(CLEANED_DATA_PATH)

print("Loaded cleaned dataset shape:", cleaned_data.shape)

print("\nColumns:")
print(cleaned_data.columns)

print("\nMissing values:")
print(cleaned_data.isnull().sum())

print("\nClass distribution:")
print(cleaned_data["label"].value_counts())

display(cleaned_data.head())

Loaded cleaned dataset shape: (39098, 3)

Columns:
Index(['content', 'clean_text', 'label'], dtype='str')

Missing values:
content       0
clean_text    0
label         0
dtype: int64

Class distribution:
label
1    21196
0    17902
Name: count, dtype: int64


,content,clean_text,label
0,Donald Trump Sends Out Embarrassing New Year’s...,donald trump sends embarrassing new year eve m...,0
1,Drunk Bragging Trump Staffer Started Russian C...,drunk bragging trump staffer started russian c...,0
2,Sheriff David Clarke Becomes An Internet Joke ...,sheriff david clarke becomes internet joke thr...,0
3,Trump Is So Obsessed He Even Has Obama’s Name ...,trump obsessed even obama name coded website i...,0
4,Pope Francis Just Called Out Donald Trump Duri...,pope francis called donald trump christmas spe...,0
